In [230]:
# Set Up 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


import os

pd.set_option('display.max_columns', None)  # Show all columns in DataFrame display

In [231]:
import pandas as pd
SENSOR_DATA_PATH = '../Data/RawData/Trial_2_Sensor_Data.xlsx'
DAY_SENSOR_DATA_PATH = '../Data/RawData/Trial_2_Day_Sensor_Data.xlsx'
NIGHT_SENSOR_DATA_PATH = '../Data/RawData/Trial_2_Night_Sensor_Data.xlsx'
NUTRIENT_TEMP_SENSOR_DATA_PATH = '../Data/RawData/Trial_2_Nutrient_Temperature_Sensor_Data.xlsx' # atlas pt 1000 missing from other sensor data
LETTUCE_FW_DATA_PATH = '../Data/RawData/Trial_2_Lettuce_FW_Data_Per_Plant.csv'

def load_sensor_data():
    return pd.read_excel(SENSOR_DATA_PATH)

def load_day_sensor_data():
    return pd.read_excel(DAY_SENSOR_DATA_PATH)

def load_night_sensor_data():
    return pd.read_excel(NIGHT_SENSOR_DATA_PATH)

def load_nutrient_temp_sensor_data():
    return pd.read_excel(NUTRIENT_TEMP_SENSOR_DATA_PATH)

def load_lettuce_fw_data():
    return pd.read_csv(LETTUCE_FW_DATA_PATH)


In [232]:
# features we are keeping 
# #    'carbon_dioxide': 'blue',
#     'temp_env': 'orange',
#     'humidity': 'green',
#     'pressure': 'red',
#     'electrical_conductivity': 'purple',
#     'volume': 'brown',
#     'temp_nutr': 'pink',
#     'volume_flow_rate': 'gray',
#     'weight_lag_1': 'cyan',
#     'weight_lag_2': 'magenta'

In [233]:
sensor_data = load_sensor_data()
sensor_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7894 entries, 0 to 7893
Data columns (total 8 columns):
 #   Column                                                     Non-Null Count  Dtype         
---  ------                                                     --------------  -----         
 0   DateTime                                                   7894 non-null   datetime64[ns]
 1   Atlas pH (CH0, Ion Concentration, pH)                      6878 non-null   float64       
 2   Atlas CO2 (Carbon Dioxide Gas) (CH0, Carbon Dioxide, ppm)  5652 non-null   float64       
 3   Atlas EC (CH0, Electrical Conductivity, μS/cm)             7580 non-null   float64       
 4   BME280 (CH0, Temperature, °C)                              4989 non-null   float64       
 5   BME280 (CH1, Humidity, %)                                  5718 non-null   float64       
 6   BME280 (CH5, Vapor Pressure Deficit, Pa)                   4429 non-null   float64       
 7   Atlas PT-1000 (CH0, Temperature, °

In [234]:
sensor_data

,DateTime,"Atlas pH (CH0, Ion Concentration, pH)","Atlas CO2 (Carbon Dioxide Gas) (CH0, Carbon Dioxide, ppm)","Atlas EC (CH0, Electrical Conductivity, μS/cm)","BME280 (CH0, Temperature, °C)","BME280 (CH1, Humidity, %)","BME280 (CH5, Vapor Pressure Deficit, Pa)","Atlas PT-1000 (CH0, Temperature, °C)"
0,2026-01-19 19:20:36,NaN,0.27,1376.0,22.342666,57.788509,1139.319637,23.7120
1,2026-01-19 19:23:20,NaN,0.27,1468.0,22.596074,53.182916,1283.050990,23.7580
2,2026-01-19 19:26:04,NaN,0.28,1457.5,22.380677,56.845071,1166.931412,23.7660
3,2026-01-19 19:28:48,NaN,0.27,1460.0,22.545392,53.419342,1272.823989,23.7160
4,2026-01-19 19:31:32,5.780,0.28,1465.0,22.246371,56.459571,1168.430102,23.6975
...,...,...,...,...,...,...,...,...
7889,2026-02-16 23:47:52,6.191,556.00,1417.0,20.173522,87.780906,NaN,21.6520
7890,2026-02-16 23:50:36,6.200,553.50,1412.0,20.665123,85.817448,345.456857,21.5840
7891,2026-02-16 23:53:20,6.250,546.50,1438.5,20.763950,85.052202,366.401535,21.5320
7892,2026-02-16 23:56:04,6.221,541.00,1424.0,19.697129,89.523473,NaN,21.4460


In [235]:
# rename columns and drop correlated features
sensor_data.drop(columns=['Atlas pH (CH0, Ion Concentration, pH)',
                          ], inplace=True)
sensor_data.rename(columns={'DateTime':'time_stamp',
                            'Atlas CO2 (Carbon Dioxide Gas) (CH0, Carbon Dioxide, ppm)': 'carbon_dioxide',
                          'Atlas EC (CH0, Electrical Conductivity, μS/cm)': 'electrical_conductivity',
                          'BME280 (CH0, Temperature, °C)': 'temp_env',
                          'Atlas PT-1000 (CH0, Temperature, °C)': 'temp_nutr',
                          'BME280 (CH1, Humidity, %)': 'humidity',
                          'BME280 (CH5, Vapor Pressure Deficit, Pa)':'pressure'}, inplace=True)



sensor_data

,time_stamp,carbon_dioxide,electrical_conductivity,temp_env,humidity,pressure,temp_nutr
0,2026-01-19 19:20:36,0.27,1376.0,22.342666,57.788509,1139.319637,23.7120
1,2026-01-19 19:23:20,0.27,1468.0,22.596074,53.182916,1283.050990,23.7580
2,2026-01-19 19:26:04,0.28,1457.5,22.380677,56.845071,1166.931412,23.7660
3,2026-01-19 19:28:48,0.27,1460.0,22.545392,53.419342,1272.823989,23.7160
4,2026-01-19 19:31:32,0.28,1465.0,22.246371,56.459571,1168.430102,23.6975
...,...,...,...,...,...,...,...
7889,2026-02-16 23:47:52,556.00,1417.0,20.173522,87.780906,NaN,21.6520
7890,2026-02-16 23:50:36,553.50,1412.0,20.665123,85.817448,345.456857,21.5840
7891,2026-02-16 23:53:20,546.50,1438.5,20.763950,85.052202,366.401535,21.5320
7892,2026-02-16 23:56:04,541.00,1424.0,19.697129,89.523473,NaN,21.4460


In [236]:
def get_4_day_bin(entry):
    if entry == 0:
        return 0
    elif entry in {1,2,3,4}:
        return 4
    elif entry in {5,6,7,8}:
        return 8
    elif entry in {9,10,11,12}:
        return 12
    elif entry in {13,14,15,16}:
        return 16
    elif entry in {17,18,19,20}:
        return 20
    elif entry in {21,22,23,24}:
        return 24
    elif entry in {25,26,27,28}:
        return 28
    

def add_day_time_bins(df, time_col='time_stamp'):
    df = df.copy()
    day_0_day = df[time_col].dt.floor('D').min()  # Get the start day
    df['temp_day'] = (df[time_col].dt.floor('D') - day_0_day).dt.days
    df['day_period'] = df['temp_day'].apply(get_4_day_bin)
    df.drop(columns=['temp_day'], inplace=True)

    # drop rows with day_period nan (since they are outside 28 day period)
    df.dropna(subset=['day_period'], inplace=True)
  
    return df

sensor_data = add_day_time_bins(sensor_data)

sensor_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7894 entries, 0 to 7893
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   time_stamp               7894 non-null   datetime64[ns]
 1   carbon_dioxide           5652 non-null   float64       
 2   electrical_conductivity  7580 non-null   float64       
 3   temp_env                 4989 non-null   float64       
 4   humidity                 5718 non-null   float64       
 5   pressure                 4429 non-null   float64       
 6   temp_nutr                6788 non-null   float64       
 7   day_period               7894 non-null   int64         
dtypes: datetime64[ns](1), float64(6), int64(1)
memory usage: 493.5 KB


In [ ]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

numeric_features = sensor_data.columns.drop(['time_stamp', 'day_period'])

# standardize data
scaler = StandardScaler()
sensor_data_scaled = sensor_data.copy()
sensor_data_scaled[numeric_features] = scaler.fit_transform(sensor_data_scaled[numeric_features])

# initialize knn regressor for imputation
knn_regressor = KNeighborsRegressor(n_neighbors=5, weights = 'distance')
# initialize knn imputer with regressor
knn_imputer = KNNImputer(n_neighbors=5, weights = 'distance')

# sort feature names by missing values (least missing to most missing)
missing_features_sorted = (sensor_data[numeric_features]
                                        .isna().sum()
                                        .sort_values(ascending=True)
                                        .index
                                        .to_list()
                           )

for feature_to_impute in missing_features_sorted:
    features_to_fit_on = [feat for feat in missing_features_sorted if feat != feature_to_impute]

    # impute for regression
    temp_df = sensor_data_scaled.copy()

    observed_mask = temp_df[feature_to_impute].notna()
    missing_mask = temp_df[feature_to_impute].isna()

    # fill missing values to prepare dataframe for knn regression
    temp_df[features_to_fit_on]= knn_imputer.fit_transform(temp_df[features_to_fit_on])
    
    # fit knn regressor 
    X_train = temp_df.loc[observed_mask, features_to_fit_on] # records to train regressor on (only observed values for feature we are imputing)
    y_train = temp_df.loc[observed_mask, feature_to_impute] # target variable for regressor (observed values for feature we are imputing)

    X_missing = temp_df.loc[missing_mask, features_to_fit_on] # records to predict on (only missing values for feature we are imputing)

    knn_regressor.fit(X_train, y_train)

    # impute data with knn regressor
    sensor_data_scaled.loc[missing_mask,feature_to_impute] = knn_regressor.predict(X_missing)
    
    print(features_to_fit_on, feature_to_impute)

sensor_data_imputed = sensor_data_scaled.copy()
sensor_data_imputed.info()



['temp_nutr', 'humidity', 'carbon_dioxide', 'temp_env', 'pressure'] electrical_conductivity
['electrical_conductivity', 'humidity', 'carbon_dioxide', 'temp_env', 'pressure'] temp_nutr
['electrical_conductivity', 'temp_nutr', 'carbon_dioxide', 'temp_env', 'pressure'] humidity
['electrical_conductivity', 'temp_nutr', 'humidity', 'temp_env', 'pressure'] carbon_dioxide
['electrical_conductivity', 'temp_nutr', 'humidity', 'carbon_dioxide', 'pressure'] temp_env
['electrical_conductivity', 'temp_nutr', 'humidity', 'carbon_dioxide', 'temp_env'] pressure
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7894 entries, 0 to 7893
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   time_stamp               7894 non-null   datetime64[ns]
 1   carbon_dioxide           7894 non-null   float64       
 2   electrical_conductivity  7894 non-null   float64       
 3   temp_env                 7894 no

save csv 

In [238]:

df_mean_sensor = sensor_data_imputed.copy()
df_mean_sensor.drop(columns={'time_stamp'}, inplace=True)
#average sensor data per 4 day period
df_mean_sensor = df_mean_sensor.groupby('day_period')[numeric_features].mean()
df_mean_sensor


,carbon_dioxide,electrical_conductivity,temp_env,humidity,pressure,temp_nutr
day_period,,,,,,
0,-1.550026,-0.623293,-0.508265,0.766095,-0.784530,0.050330
4,-0.399247,-0.008630,0.066717,-0.059020,-0.155939,0.441230
8,-0.510252,-0.347143,0.091212,0.091056,-0.222485,1.131556
12,1.234844,1.261565,0.368473,-0.216415,-0.132565,0.144733
16,0.134025,-0.245045,0.425484,-0.370781,0.022822,-0.333579
20,0.002274,-0.309950,0.221699,-0.034513,-0.379621,-0.581299
24,-0.324009,-0.137120,0.025168,0.135163,-0.609285,-0.352628
28,0.268913,-0.432599,0.113866,0.336439,-0.858064,-0.735182


In [239]:
lettuce_fw_data = load_lettuce_fw_data()
lettuce_fw_data

,Date,Day,Plant-ID,Total Fresh Weight (g),Baseline (g),New Fresh Weight (g),Median Fresh Weight (g),Average Fresh Weight (g)
0,2026-01-19,0.0,6.0,29.1,25.22,3.9,NaN,NaN
1,NaN,0.0,4.0,29.3,25.22,4.1,NaN,NaN
2,NaN,0.0,9.0,29.6,25.22,4.4,NaN,NaN
3,NaN,0.0,1.0,29.7,25.22,4.5,NaN,NaN
4,NaN,0.0,8.0,29.8,25.22,4.6,NaN,NaN
...,...,...,...,...,...,...,...,...
104,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
105,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
106,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
107,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [240]:
lettuce_fw_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109 entries, 0 to 108
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Date                      8 non-null      object 
 1   Day                       80 non-null     float64
 2   Plant-ID                  80 non-null     float64
 3   Total Fresh Weight (g)    80 non-null     float64
 4   Baseline (g)              80 non-null     float64
 5   New Fresh Weight (g)      80 non-null     float64
 6   Median Fresh Weight (g)   8 non-null      float64
 7   Average Fresh Weight (g)  8 non-null      float64
dtypes: float64(7), object(1)
memory usage: 6.9+ KB


In [241]:
#lettuce_fw_data.info()
#  <class 'pandas.core.frame.DataFrame'>
# RangeIndex: 109 entries, 0 to 108
# Data columns (total 8 columns):
#  #   Column                    Non-Null Count  Dtype  
# ---  ------                    --------------  -----  
#  0   Date                      8 non-null      object 
#  1   Day                       80 non-null     float64
#  2   Plant-ID                  80 non-null     float64
#  3   Total Fresh Weight (g)    80 non-null     float64
#  4   Baseline (g)              80 non-null     float64
#  5   New Fresh Weight (g)      80 non-null     float64
#  6   Median Fresh Weight (g)   8 non-null      float64
#  7   Average Fresh Weight (g)  8 non-null      float64
# dtypes: float64(7), object(1)
# memory usage: 6.9+ KB

In [242]:
# extract features that will be joined with sensor data 

df_lettuce_fw = lettuce_fw_data[['Day', 'Plant-ID', 'Total Fresh Weight (g)']]
df_lettuce_fw.rename(columns={'Day':'day', 'Plant-ID':'plant_id', 'Total Fresh Weight (g)': 'total_fresh_weight_g'}, inplace=True)
df_lettuce_fw.dropna(how='all', inplace=True)
df_lettuce_fw.info()

<class 'pandas.core.frame.DataFrame'>
Index: 80 entries, 0 to 86
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   day                   80 non-null     float64
 1   plant_id              80 non-null     float64
 2   total_fresh_weight_g  80 non-null     float64
dtypes: float64(3)
memory usage: 2.5 KB


/var/folders/b2/dv4s0x994dz37s4d7fv56q8r0000gn/T/ipykernel_2949/173458327.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_lettuce_fw.rename(columns={'Day':'day', 'Plant-ID':'plant_id', 'Total Fresh Weight (g)': 'total_fresh_weight_g'}, inplace=True)
/var/folders/b2/dv4s0x994dz37s4d7fv56q8r0000gn/T/ipykernel_2949/173458327.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_lettuce_fw.dropna(how='all', inplace=True)


In [243]:
df_lettuce_fw

,day,plant_id,total_fresh_weight_g
0,0.0,6.0,29.1
1,0.0,4.0,29.3
2,0.0,9.0,29.6
3,0.0,1.0,29.7
4,0.0,8.0,29.8
...,...,...,...
82,28.0,2.0,206.0
83,28.0,4.0,208.0
84,28.0,6.0,211.0
85,28.0,8.0,215.0


In [244]:
# 

Join both dataframes on day_period and day and save as csv.

In [245]:
df_mean_sensor_and_lettuce_fw_4days = pd.merge(df_mean_sensor, df_lettuce_fw,
                                               left_on='day_period',
                                                right_on='day',
                                                how='inner'
)

# reorder columns to have day, plant_id, total_fresh_weight at the beginning
df_mean_sensor_and_lettuce_fw_4days = df_mean_sensor_and_lettuce_fw_4days[['day', 'plant_id', 'total_fresh_weight_g', 'carbon_dioxide', 'electrical_conductivity', 'temp_env', 'temp_nutr', 'humidity', 'pressure']]
df_mean_sensor_and_lettuce_fw_4days.sort_values(by=['day', 'plant_id'], inplace=True)
df_mean_sensor_and_lettuce_fw_4days.reset_index(drop=True, inplace=True)
df_mean_sensor_and_lettuce_fw_4days

,day,plant_id,total_fresh_weight_g,carbon_dioxide,electrical_conductivity,temp_env,temp_nutr,humidity,pressure
0,0.0,1.0,29.7,-1.550026,-0.623293,-0.508265,0.050330,0.766095,-0.784530
1,0.0,2.0,30.1,-1.550026,-0.623293,-0.508265,0.050330,0.766095,-0.784530
2,0.0,3.0,31.8,-1.550026,-0.623293,-0.508265,0.050330,0.766095,-0.784530
3,0.0,4.0,29.3,-1.550026,-0.623293,-0.508265,0.050330,0.766095,-0.784530
4,0.0,5.0,30.2,-1.550026,-0.623293,-0.508265,0.050330,0.766095,-0.784530
...,...,...,...,...,...,...,...,...,...
75,28.0,6.0,211.0,0.268913,-0.432599,0.113866,-0.735182,0.336439,-0.858064
76,28.0,7.0,188.0,0.268913,-0.432599,0.113866,-0.735182,0.336439,-0.858064
77,28.0,8.0,215.0,0.268913,-0.432599,0.113866,-0.735182,0.336439,-0.858064
78,28.0,9.0,198.0,0.268913,-0.432599,0.113866,-0.735182,0.336439,-0.858064


download clean dataframe and save as csv

In [246]:
Trial_2_Average_Sensor_Data_Per_4_Day_Period = df_mean_sensor_and_lettuce_fw_4days.copy()
Trial_2_Average_Sensor_Data_Per_4_Day_Period.to_csv('../Data/CleanData/Trial_2_Average_Sensor_LettuceFW_Per_4_Day_Period.csv', index=False)

add lag variables to clean dataframe and save 3 dataframes as csv

In [247]:
df_lagged = df_mean_sensor_and_lettuce_fw_4days.copy()
df_lagged['weight_lag_1'] = (df_lagged.groupby('plant_id')['total_fresh_weight_g'].shift(1))
df_lagged['weight_lag_2'] = (df_lagged.groupby('plant_id')['total_fresh_weight_g'].shift(2))

Trial_2_Mean_Sensor_FW_No_Lag = df_mean_sensor_and_lettuce_fw_4days.copy()
Trial_2_Mean_Sensor_FW_Lag_1 = (df_lagged.drop(columns=['weight_lag_2']).copy()
                                .dropna(subset=['weight_lag_1'])
                                )
Trial_2_Mean_Sensor_FW_Lag_2 = (df_lagged.copy()
                                .dropna(subset=['weight_lag_1', 'weight_lag_2'])
                                )

Trial_2_Mean_Sensor_FW_No_Lag.to_csv('../Data/CleanData/Trial_2_Mean_Sensor_FW_No_Lag.csv', index=False)
Trial_2_Mean_Sensor_FW_Lag_1.to_csv('../Data/CleanData/Trial_2_Mean_Sensor_FW_Lag_1.csv', index=False)
Trial_2_Mean_Sensor_FW_Lag_2.to_csv('../Data/CleanData/Trial_2_Mean_Sensor_FW_Lag_2.csv', index=False)
